# Diagnostics CSV Overview

Quick visual overview for `evaluation_mismatch_diagnostics.csv`: distributions, issue tables, dataset/model splits, and review samples.

In [ ]:
from pathlib import Path
import json

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_colwidth", 180)
pd.set_option("display.max_rows", 200)

def find_repo_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    for path in [start, *start.parents]:
        if (path / "src").exists() and (path / "results").exists():
            return path
    raise FileNotFoundError("Could not find repo root containing src/ and results/.")

ROOT = find_repo_root()
DIAG_PATH = ROOT / "results" / "experiment" / "nl2p_1" / "gpt-4o" / "evaluation_mismatch_diagnostics.csv"
DIAG_PATH

In [ ]:
df = pd.read_csv(DIAG_PATH)

text_cols = [
    "candidate_dataset_issue",
    "candidate_llm_issue",
    "reason",
    "gold_verb",
    "pred_verb",
    "gold_arguments",
    "pred_arguments",
]
for col in text_cols:
    if col in df.columns:
        df[col] = df[col].fillna("")

def parse_args(value):
    if not isinstance(value, str) or not value.strip():
        return []
    try:
        parsed = json.loads(value)
    except json.JSONDecodeError:
        return []
    return parsed if isinstance(parsed, list) else []

df["gold_arg_list"] = df["gold_arguments"].map(parse_args)
df["pred_arg_list"] = df["pred_arguments"].map(parse_args)
df["gold_arg_count"] = df["gold_arg_list"].map(len)
df["pred_arg_count"] = df["pred_arg_list"].map(len)
df["has_dataset_issue"] = df["candidate_dataset_issue"].ne("")
df["has_llm_issue"] = df["candidate_llm_issue"].ne("")

print(f"rows: {len(df):,}")
print(f"datasets: {sorted(df['dataset'].dropna().unique())}")
print(f"mismatch types: {sorted(df['mismatch_type'].dropna().unique())}")
df.head(3)

In [ ]:
summary = (
    df.groupby(["dataset", "mismatch_type"], dropna=False)
    .agg(
        rows=("mismatch_type", "size"),
        dataset_issue_rows=("has_dataset_issue", "sum"),
        llm_issue_rows=("has_llm_issue", "sum"),
        avg_gold_args=("gold_arg_count", "mean"),
        avg_pred_args=("pred_arg_count", "mean"),
    )
    .reset_index()
)
summary

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

sns.countplot(data=df, y="mismatch_type", order=df["mismatch_type"].value_counts().index, ax=axes[0])
axes[0].set_title("Mismatch type distribution")
axes[0].set_xlabel("Rows")
axes[0].set_ylabel("")

issue_source = pd.DataFrame({
    "source": ["dataset issue", "llm issue", "both", "neither"],
    "rows": [
        int((df["has_dataset_issue"] & ~df["has_llm_issue"]).sum()),
        int((~df["has_dataset_issue"] & df["has_llm_issue"]).sum()),
        int((df["has_dataset_issue"] & df["has_llm_issue"]).sum()),
        int((~df["has_dataset_issue"] & ~df["has_llm_issue"]).sum()),
    ],
})
sns.barplot(data=issue_source, x="rows", y="source", ax=axes[1])
axes[1].set_title("Candidate attribution rows")
axes[1].set_xlabel("Rows")
axes[1].set_ylabel("")

plt.tight_layout()

In [ ]:
def split_issue_labels(frame, column):
    rows = []
    for idx, value in frame[column].items():
        for label in str(value).split("|"):
            label = label.strip()
            if label:
                rows.append({"row_id": idx, "label": label})
    return pd.DataFrame(rows)

dataset_labels = split_issue_labels(df, "candidate_dataset_issue")
llm_labels = split_issue_labels(df, "candidate_llm_issue")

display(dataset_labels["label"].value_counts().rename_axis("dataset_issue").reset_index(name="rows"))
display(llm_labels["label"].value_counts().rename_axis("llm_issue").reset_index(name="rows"))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

top_dataset = dataset_labels["label"].value_counts().head(12)
sns.barplot(x=top_dataset.values, y=top_dataset.index, ax=axes[0])
axes[0].set_title("Top dataset-side diagnostic labels")
axes[0].set_xlabel("Rows")
axes[0].set_ylabel("")

top_llm = llm_labels["label"].value_counts().head(12)
sns.barplot(x=top_llm.values, y=top_llm.index, ax=axes[1])
axes[1].set_title("Top LLM-side diagnostic labels")
axes[1].set_xlabel("Rows")
axes[1].set_ylabel("")

plt.tight_layout()

## Focus: Gold Split and Preposition Object Coverage

This section measures the share of the two main argument-side GT issue families by domain. It reports percentages against three denominators: all diagnostic mismatch rows, argument-mismatch rows only, and dataset-side argument-issue rows.

In [ ]:
FOCUS_ISSUES = {
    "gold_split": "unnecessary_head_or_modifier_split",
    "preposition_object": "preposition_object",
    "gold_pronoun_or_generic_reference": "gold_pronoun_or_generic_reference",
    "gold_missing_valid_argument": "gold_missing_valid_argument",
}

def has_issue(frame, issue_substring):
    return frame["candidate_dataset_issue"].str.contains(issue_substring, regex=False, na=False)

def domain_coverage_table(frame):
    rows = []
    for domain, group in frame.groupby("dataset", dropna=False):
        all_rows = len(group)
        arg_rows = int(group["mismatch_type"].eq("argument_mismatch").sum())
        dataset_arg_mask = group["mismatch_type"].eq("argument_mismatch") & group["has_dataset_issue"]
        dataset_arg_rows = int(dataset_arg_mask.sum())

        for issue_name, issue_substring in FOCUS_ISSUES.items():
            mask = has_issue(group, issue_substring)
            count = int(mask.sum())
            arg_count = int((mask & group["mismatch_type"].eq("argument_mismatch")).sum())
            rows.append({
                "dataset": domain,
                "issue": issue_name,
                "rows": count,
                "arg_mismatch_rows": arg_count,
                "all_diagnostic_rows": all_rows,
                "all_argument_mismatch_rows": arg_rows,
                "dataset_side_argument_issue_rows": dataset_arg_rows,
                "pct_of_all_diagnostics": count / all_rows if all_rows else 0,
                "pct_of_argument_mismatches": arg_count / arg_rows if arg_rows else 0,
                "pct_of_dataset_side_argument_issues": arg_count / dataset_arg_rows if dataset_arg_rows else 0,
            })
    return pd.DataFrame(rows)

coverage = domain_coverage_table(df)
coverage_display = coverage.copy()
for col in ["pct_of_all_diagnostics", "pct_of_argument_mismatches", "pct_of_dataset_side_argument_issues"]:
    coverage_display[col] = (coverage_display[col] * 100).round(1)
coverage_display.sort_values(["dataset", "rows"], ascending=[True, False])

In [ ]:
plot_issues = ["gold_split", "preposition_object"]
plot_data = coverage[coverage["issue"].isin(plot_issues)].copy()
plot_data["percent"] = plot_data["pct_of_argument_mismatches"] * 100

plt.figure(figsize=(9, 4))
sns.barplot(data=plot_data, x="dataset", y="percent", hue="issue")
plt.title("Gold split / preposition object as % of argument mismatches")
plt.xlabel("Domain")
plt.ylabel("% of argument_mismatch rows")
plt.ylim(0, max(5, plot_data["percent"].max() * 1.2))
plt.legend(title="Issue family")
plt.tight_layout()

In [ ]:
plot_data = coverage[coverage["issue"].isin(plot_issues)].copy()
plot_data["percent"] = plot_data["pct_of_all_diagnostics"] * 100

plt.figure(figsize=(9, 4))
sns.barplot(data=plot_data, x="dataset", y="percent", hue="issue")
plt.title("Gold split / preposition object as % of all diagnostic rows")
plt.xlabel("Domain")
plt.ylabel("% of all diagnostics rows")
plt.ylim(0, max(5, plot_data["percent"].max() * 1.2))
plt.legend(title="Issue family")
plt.tight_layout()

In [ ]:
combined = []
for domain, group in df.groupby("dataset", dropna=False):
    arg_group = group[group["mismatch_type"].eq("argument_mismatch")]
    denom = len(arg_group)
    split_mask = has_issue(arg_group, FOCUS_ISSUES["gold_split"])
    prep_mask = has_issue(arg_group, FOCUS_ISSUES["preposition_object"])
    union_mask = split_mask | prep_mask
    combined.append({
        "dataset": domain,
        "argument_mismatch_rows": denom,
        "gold_split_rows": int(split_mask.sum()),
        "preposition_object_rows": int(prep_mask.sum()),
        "split_or_preposition_rows": int(union_mask.sum()),
        "pct_split_or_preposition_of_argument_mismatches": (union_mask.sum() / denom * 100) if denom else 0,
    })
combined = pd.DataFrame(combined)
combined["pct_split_or_preposition_of_argument_mismatches"] = combined["pct_split_or_preposition_of_argument_mismatches"].round(1)
combined

In [ ]:
if not dataset_labels.empty:
    dataset_heatmap = (
        dataset_labels.join(df[["dataset"]], on="row_id")
        .pivot_table(index="label", columns="dataset", values="row_id", aggfunc="count", fill_value=0)
    )
    dataset_heatmap = dataset_heatmap.loc[dataset_heatmap.sum(axis=1).sort_values(ascending=False).head(15).index]
    plt.figure(figsize=(8, max(4, 0.35 * len(dataset_heatmap))))
    sns.heatmap(dataset_heatmap, annot=True, fmt="d", cmap="Blues")
    plt.title("Dataset-side labels by dataset")
    plt.xlabel("Dataset")
    plt.ylabel("Label")
    plt.tight_layout()
else:
    print("No dataset-side issue labels found.")

In [ ]:
argument_rows = df[df["mismatch_type"].eq("argument_mismatch")].copy()

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
sns.histplot(data=argument_rows, x="gold_arg_count", discrete=True, ax=axes[0])
axes[0].set_title("Gold argument count in argument mismatches")
axes[0].set_xlabel("Gold argument count")

sns.histplot(data=argument_rows, x="pred_arg_count", discrete=True, ax=axes[1])
axes[1].set_title("Predicted argument count in argument mismatches")
axes[1].set_xlabel("Pred argument count")

plt.tight_layout()

pd.crosstab(argument_rows["gold_arg_count"], argument_rows["pred_arg_count"], margins=True)

In [ ]:
verb_table = (
    df.assign(pred_verb_norm=df["pred_verb"].str.lower().str.strip(), gold_verb_norm=df["gold_verb"].str.lower().str.strip())
    .groupby(["mismatch_type", "gold_verb_norm", "pred_verb_norm"], dropna=False)
    .size()
    .reset_index(name="rows")
    .sort_values("rows", ascending=False)
)
verb_table.head(40)

In [ ]:
def sample_rows(issue=None, issue_col="candidate_dataset_issue", dataset=None, mismatch_type=None, n=10, random_state=7):
    subset = df.copy()
    if issue:
        subset = subset[subset[issue_col].str.contains(issue, regex=False, na=False)]
    if dataset:
        subset = subset[subset["dataset"].eq(dataset)]
    if mismatch_type:
        subset = subset[subset["mismatch_type"].eq(mismatch_type)]
    if len(subset) > n:
        subset = subset.sample(n=n, random_state=random_state)
    cols = [
        "dataset",
        "doc_id",
        "mismatch_type",
        "candidate_dataset_issue",
        "candidate_llm_issue",
        "reason",
        "gold_verb",
        "gold_arguments",
        "pred_verb",
        "pred_arguments",
        "original_text",
    ]
    return subset[cols]

# Examples:
# sample_rows(issue="gold_pronoun_or_generic_reference", n=10)
# sample_rows(issue="gold_missing_valid_argument", n=10)
# sample_rows(issue="preposition_object", n=10)
# sample_rows(issue="unnecessary_head_or_modifier_split", n=10)
sample_rows(n=5)